In [ ]:
!pip -q install gradio
!pip -q install transformers peft accelerate sentencepiece
!pip install -U "bitsandbytes>=0.46.1" accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 42.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 383.7/383.7 kB 39.1 MB/s eta 0:00:00
  Attempting uninstall: accelerate
    Found existing installation: accelerate 1.12.0
    Uninstalling accelerate-1.12.0:
      Successfully uninstalled accelerate-1.12.0


In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

In [ ]:
# =========================
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel

BASE_MODEL = "Qwen/Qwen2.5-3B-Instruct"   # change only if you trained on a different base
ADAPTER_PATH = "/content/drive/MyDrive/llm_judge_3b_result/final_weights"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

infer_tok = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
if infer_tok.pad_token is None:
    infer_tok.pad_token = infer_tok.eos_token

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.bfloat16,
    trust_remote_code=True,
)

infer_model = PeftModel.from_pretrained(
    base_model,
    ADAPTER_PATH,
)

infer_model.eval()

config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): Qwen2ForCausalLM(
      (model): Qwen2Model(
        (embed_tokens): Embedding(151936, 2048)
        (layers): ModuleList(
          (0-35): 36 x Qwen2DecoderLayer(
            (self_attn): Qwen2Attention(
              (q_proj): lora.Linear4bit(
                (base_layer): Linear4bit(in_features=2048, out_features=2048, bias=True)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=2048, out_features=16, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=16, out_features=2048, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): lora

In [ ]:

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.bfloat16,
    trust_remote_code=True,
)

infer_model = PeftModel.from_pretrained(
    base_model,
    ADAPTER_PATH,
)

infer_model.eval()


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): Qwen2ForCausalLM(
      (model): Qwen2Model(
        (embed_tokens): Embedding(151936, 2048)
        (layers): ModuleList(
          (0-35): 36 x Qwen2DecoderLayer(
            (self_attn): Qwen2Attention(
              (q_proj): lora.Linear4bit(
                (base_layer): Linear4bit(in_features=2048, out_features=2048, bias=True)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=2048, out_features=16, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=16, out_features=2048, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): lora

In [ ]:
SYSTEM_PROMPT = (
    "You are a summarization judge. "
    "Evaluate the summary against the source document on four dimensions: "
    "relevance, coherence, fluency, and consistency. "
    "Return ONLY a single valid JSON object with NO extra text before or after. "
    "The JSON must have exactly these keys: "
    "relevance, coherence, fluency, consistency, explanations. "
    "The first four must be integers 1-5. "
    "'explanations' must be a JSON object with exactly these string keys: "
    "relevance, coherence, fluency, consistency. "
    "Example format:\n"
    '{"relevance": 3, "coherence": 4, "fluency": 5, "consistency": 3, '
    '"explanations": {"relevance": "...", "coherence": "...", "fluency": "...", "consistency": "..."}}'
)

def make_user_prompt(doc, summary):
    return (
        f"Document:\n{doc}\n\n"
        f"Summary:\n{summary}\n\n"
        "Evaluate the summary and return JSON only."
    )
test_rows = []
import json
import re

def safe_parse_json(text):
    if text is None:
        return None

    text = text.strip()

    # try direct parse first
    try:
        return json.loads(text)
    except Exception:
        pass

    # extract first {...} block
    m = re.search(r'\{.*\}', text, flags=re.DOTALL)
    if m:
        candidate = m.group(0)
        try:
            return json.loads(candidate)
        except Exception:
            pass

    return None
import re

def extract_scores_and_reasoning_regex(text):
    out = {
        "pred_relevance": None,
        "pred_coherence": None,
        "pred_fluency": None,
        "pred_consistency": None,
        "pred_relevance_expl": None,
        "pred_coherence_expl": None,
        "pred_fluency_expl": None,
        "pred_consistency_expl": None,
    }

    if text is None:
        return out

    score_patterns = {
        "pred_relevance": r'"relevance"\s*:\s*([1-5])',
        "pred_coherence": r'"coherence"\s*:\s*([1-5])',
        "pred_fluency": r'"fluency"\s*:\s*([1-5])',
        "pred_consistency": r'"consistency"\s*:\s*([1-5])',
    }
    for k, p in score_patterns.items():
        m = re.search(p, text)
        if m:
            out[k] = int(m.group(1))

    expl_block = None
    for pattern in [
        r'"?explanations"?\s*:?\s*\{(.*?)\}\s*$',
        r'"explanations"\s*:\s*\{(.*)',
        r'explanations\s*\{(.*)',
    ]:
        m_block = re.search(pattern, text, flags=re.DOTALL)
        if m_block:
            expl_block = m_block.group(1)
            break

    if expl_block is not None:
        expl_patterns = {
            "relevance": r'"relevance"\s*:\s*"((?:[^"\\]|\\.)*)"',
            "coherence": r'"coherence"\s*:\s*"((?:[^"\\]|\\.)*)"',
            "fluency":   r'"fluency"\s*:\s*"((?:[^"\\]|\\.)*)"',
            "consistency": r'"consistency"\s*:\s*"((?:[^"\\]|\\.)*)"',
        }
        for k, p in expl_patterns.items():
            m = re.search(p, expl_block, flags=re.DOTALL)
            if m:
                val = m.group(1)
                val = val.replace('\\"', '"').replace("\\n", "\n").replace("\\t", "\t")
                out[f"pred_{k}_expl"] = val.strip()  # ← fix here

    return out

def judge_one_from_prompt(user_prompt, max_new_tokens=386):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_prompt},
    ]
    text = infer_tok.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )
    inputs = infer_tok(text, return_tensors="pt").to(infer_model.device)

    with torch.no_grad():
        out = infer_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            # temperature=0.0,
            # top_p=1.0,
            pad_token_id=infer_tok.eos_token_id,
            eos_token_id=infer_tok.eos_token_id,
        )

    gen = infer_tok.decode(
        out[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True
    ).strip()
    return gen


In [ ]:
a = "Yes, there are other search engines but for the majority of internet users, to search is to Google. Now, though, there are signs of a threat to to the Californian firm's hegemony."

b = "there are other search engines"
p = make_user_prompt(a, b)
print(p)
pred_text = judge_one_from_prompt(p, max_new_tokens=320)
print(pred_text)
parsed = extract_scores_and_reasoning_regex(pred_text)
parsed

Document:
Yes, there are other search engines but for the majority of internet users, to search is to Google. Now, though, there are signs of a threat to to the Californian firm's hegemony.

Summary:
there are other search engines

Evaluate the summary and return JSON only.
{"relevance": 2, "coherence": 4, "fluency": 5, "consistency": 2, "explanations": {"relevance": "The summary captures the basic idea that there are alternatives but omits key details about Google's dominant position and signs of threat.", "coherence": "The single sentence is logically organized and easy to follow."}, "fluency": "The summary is clear and readable with no awkward phrasing."}, "consistency": "The statement is accurate in its core claim but it is incomplete and omits important context from the source."}}


{'pred_relevance': 2,
 'pred_coherence': 4,
 'pred_fluency': 5,
 'pred_consistency': 2,
 'pred_relevance_expl': "The summary captures the basic idea that there are alternatives but omits key details about Google's dominant position and signs of threat.",
 'pred_coherence_expl': 'The single sentence is logically organized and easy to follow.',
 'pred_fluency_expl': 'The summary is clear and readable with no awkward phrasing.',
 'pred_consistency_expl': 'The statement is accurate in its core claim but it is incomplete and omits important context from the source.'}

In [ ]:
import gradio as gr
def demo_fn(source_text, summary_text):
    p = make_user_prompt(source_text, summary_text)
    pred_text = judge_one_from_prompt(p, max_new_tokens=256)
    parsed = extract_scores_and_reasoning_regex(pred_text)

    return (
        parsed["pred_coherence"],
        parsed["pred_consistency"],
        parsed["pred_fluency"],
        parsed["pred_relevance"],
        parsed["pred_coherence_expl"],
        parsed["pred_consistency_expl"],
        parsed["pred_fluency_expl"],
        parsed["pred_relevance_expl"],
        pred_text
    )




with gr.Blocks() as demo:
    gr.Markdown("# Summary Judge Demo")

    with gr.Row():
        source_input = gr.Textbox(label="Source Article", lines=16)
        summary_input = gr.Textbox(label="Candidate Summary", lines=16)

    run_btn = gr.Button("Evaluate")

    with gr.Row():
        coherence_out = gr.Number(label="Coherence")
        consistency_out = gr.Number(label="Consistency")
        fluency_out = gr.Number(label="Fluency")
        relevance_out = gr.Number(label="Relevance")

    coherence_reason = gr.Textbox(label="Coherence Reasoning", lines=4)
    consistency_reason = gr.Textbox(label="Consistency Reasoning", lines=4)
    fluency_reason = gr.Textbox(label="Fluency Reasoning", lines=4)
    relevance_reason = gr.Textbox(label="Relevance Reasoning", lines=4)

    raw_out = gr.Textbox(label="Raw Output", lines=10)

    run_btn.click(
        fn=demo_fn,
        inputs=[source_input, summary_input],
        outputs=[
            coherence_out,
            consistency_out,
            fluency_out,
            relevance_out,
            coherence_reason,
            consistency_reason,
            fluency_reason,
            relevance_reason,
            raw_out
        ]
    )

demo.launch(share=True, debug=True)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://68ae0088cab85b3678.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


webinterface